In [9]:
# Step 1 – Initial data loading and structural inspection
# Goal: validate schema, column types, and overall data integrity before cleaning

import pandas as pd
import numpy as np

df = pd.read_csv('data/supermarket_sales_dirty.csv')

df.shape
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Invoice ID     1005 non-null   str    
 1   Branch         1005 non-null   str    
 2   Customer type  1005 non-null   str    
 3   Gender         1005 non-null   str    
 4   Product line   1005 non-null   str    
 5   Unit price     955 non-null    float64
 6   Quantity       956 non-null    str    
 7   Date           1005 non-null   str    
 8   Time           1005 non-null   str    
 9   Payment        1005 non-null   str    
 10  Rating         955 non-null    float64
 11  City           1005 non-null   str    
 12  Total          1005 non-null   float64
dtypes: float64(3), str(10)
memory usage: 102.2 KB


In [10]:
# Step 2 – Classify columns by analytical role
# Goal: define validation strategy based on data function

# Identifier columns (not used for statistical analysis)
id_cols = ['Invoice ID']

# Datetime fields (require logical consistency checks)
datetime_cols = ['Date', 'Time']

# Core numeric metrics (require type validation and boundary checks)
core_numeric_cols = [
    'Unit price',     # transaction-level price
    'Quantity',     # purchase quantity
    'Total',     # transaction revenue
    'Rating'     # customer score (requires range validation)
]

# Categorical dimensions (used for grouping and segmentation)
categorical_cols = [
    'Branch',
    'City',
    'Customer type',
    'Gender',
    'Product line',
    'Payment'
]

In [11]:
# Step 3 – Enforce numeric types and expose invalid entries

# Convert core numeric fields and surface hidden parsing errors as NaN
for col in core_numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Verify data types after conversion
df[core_numeric_cols].dtypes

# Check proportion of missing values introduced during coercion
for col in core_numeric_cols:
    na_ratio = df[col].isna().mean()
    print(col, f'{na_ratio:.1%} NaN')

Unit price 5.0% NaN
Quantity 5.9% NaN
Total 0.0% NaN
Rating 5.0% NaN


In [12]:
# Step 4 – Business rule validation for numeric fields

# Check non-positive values for transaction metrics
print('Unit price <= 0:', (df['Unit price'] <= 0).sum())
print('Quantity <= 0:', (df['Quantity'] <= 0).sum())
print('Total <= 0:', (df['Total'] <= 0).sum())

# Validate rating range (expected 0–10)
print('Rating < 0:', (df['Rating'] < 0).sum())
print('Rating > 10:', (df['Rating'] > 10).sum())

Unit price <= 0: 0
Quantity <= 0: 0
Total <= 0: 0
Rating < 0: 0
Rating > 10: 0


In [13]:
# Step 5 – Missing value handling strategy (field-type based)

# Drop rows where core numeric fields contain NaN
df_clean = df.dropna(subset=core_numeric_cols).copy()

# Compare dataset size before and after cleaning
print('Original shape:', df.shape)
print('Cleaned shape:', df_clean.shape)

# Calculate removal ratio (percentage of rows removed)
removal_ratio = 1 - len(df_clean) / len(df)
print(f'Removed ratio: {removal_ratio:.1%}')

Original shape: (1005, 13)
Cleaned shape: (856, 13)
Removed ratio: 14.8%


In [14]:
# Step 6 – Categorical value consistency check (example: Payment)

# Check category distribution
print(df_clean['Payment'].value_counts())

# Minimal correction: remove leading/trailing spaces
df_clean.loc[:, 'Payment'] = df_clean['Payment'].str.strip()

# Re-check after cleaning
print(df_clean['Payment'].value_counts())

# Confirm number of unique categories (consistency check)
print('Unique category count:', df_clean['Payment'].nunique())

Payment
Cash           294
Ewallet        282
Credit card    280
Name: count, dtype: int64
Payment
Cash           294
Ewallet        282
Credit card    280
Name: count, dtype: int64
Unique category count: 3


In [15]:
# Step 7 – Datetime parsing validation

# Attempt to parse Date and Time fields
df_clean.loc[:, 'Date_parsed'] = pd.to_datetime(
    df_clean['Date'],
    errors='coerce'
)

df_clean.loc[:, 'Time_parsed'] = pd.to_datetime(
    df_clean['Time'],
    format='%H:%M',
    errors='coerce'
).dt.time

# Calculate parsing failure ratios
date_parse_fail = df_clean['Date_parsed'].isna().mean()
time_parse_fail = df_clean['Time_parsed'].isna().mean()

print(f'Date parse failure ratio: {date_parse_fail:.2%}')
print(f'Time parse failure ratio: {time_parse_fail:.2%}')

Date parse failure ratio: 0.00%
Time parse failure ratio: 0.00%


In [16]:
# Step 8 – Final sanity check after cleaning

# 1️⃣ Descriptive statistics for core numeric fields
# Confirm no abnormal negative values or unexpected ranges remain
print(df_clean[core_numeric_cols].describe())

# 2️⃣ Check remaining missing values in critical fields
print(df_clean[core_numeric_cols].isna().sum())

# 3️⃣ Compare dataset size before and after cleaning
print("Original shape:", df.shape)
print("Cleaned shape :", df_clean.shape)
print(f'Removed ratio: {removal_ratio:.1%}')

        Unit price    Quantity        Total      Rating
count   856.000000  856.000000   856.000000  856.000000
mean     58.071414    5.601636   315.936904    7.006308
std      73.019307    2.910589   244.523448    1.722532
min      10.020000    1.000000    11.410000    4.000000
25%      30.937500    3.000000   118.650000    5.500000
50%      52.480000    6.000000   244.385000    7.000000
75%      76.847500    8.000000   448.940000    8.600000
max    1052.300000   10.000000  1046.750000   10.000000
Unit price    0
Quantity      0
Total         0
Rating        0
dtype: int64
Original shape: (1005, 13)
Cleaned shape : (856, 15)
Removed ratio: 14.8%
